# Polymarket Tahmin Piyasası Arbitraj Motoru ve Kelly Kriteri Optimizasyonu

**Projenin Amacı:** 
Bu proje, Polymarket gibi tahmin piyasalarındaki (prediction markets) anlık fiyat uyuşmazlıklarını (arbitraj) tespit eden ve bulunan fırsatlara istatistiksel olarak ne kadar sermaye ayrılması gerektiğini hesaplayan algoritmik bir modeldir. 

**Teori ve Gerçek Dünya Maliyetleri:**
Piyasalarda "EVET" ve "HAYIR" kontratlarının toplamı teorik olarak her zaman 1.00$ olmalıdır. Ancak düşük likidite veya ani haber akışları sırasında bu toplam 1 doların altına düşebilir. Bu motor, sadece brüt kârı tespit etmekle kalmaz, aynı zamanda işlem komisyonu (fee) ve fiyat kayması (slippage) gibi gerçek dünya maliyetlerini de hesaba katarak net kârı hesaplar.

Net kâr tespit edildiğinde ise **Kelly Kriteri** kullanarak iflas riskini (Risk of Ruin) sıfıra indirir ve kârı maksimize edecek optimal pozisyon büyüklüğünü belirler. Böylece risk/ödül oranı negatif veya uzun vadede sürdürülemez olan fırsatları matematiksel olarak reddeder.

In [ ]:
import requests
import json

# API Bağlantısı: Aktif ve işleme açık 10 etkinliği çekiyoruz
url = "https://gamma-api.polymarket.com/events?limit=10&active=true&closed=false"
cevap = requests.get(url)
veriler = cevap.json()

# Gerçek Dünya İşlem Maliyeti Varsayımları
FEE_RATE = 0.01      # %1 Platform Komisyonu
SLIPPAGE = 0.005     # %0.5 Emir Gönderme Sırasında Oluşabilecek Fiyat Kayması

def kelly_kriteri(net_kar, toplam_maliyet, kazanma_olasiligi=0.95):
    """
    Kelly Kriteri, yatırılacak optimal sermaye oranını hesaplar.
    p = Kazanma olasılığı (Emrin gerçekleşmeme/reddedilme riskleri %5 olarak varsayılmıştır)
    q = Kaybetme olasılığı (1 - p)
    b = Risk/Ödül oranı (Net Kâr / Toplam Maliyet)
    """
    b = net_kar / toplam_maliyet
    p = kazanma_olasiligi
    q = 1.0 - p 
    
    f_star = (p * b - q) / b
    return max(0, f_star)

print("--- Maliyet Düşülmüş Net Fiyat Taraması ---")

# JSON İçinde Döngü ve Filtreleme
for etkinlik in veriler:
    baslik = etkinlik.get('title', 'Başlık Yok')
    for piyasa in etkinlik.get('markets', []):
        outcomes = json.loads(piyasa.get('outcomes', '[]'))
        prices = json.loads(piyasa.get('outcomePrices', '[]'))
        
        # Dengesiz/boş verileri filtrele ve sadece 'Yes/No' piyasalarını al
        if len(outcomes) == 2 and len(prices) >= 2 and "Yes" in outcomes and "No" in outcomes:
            yes_maliyeti = float(prices[0]) * (1 + FEE_RATE + SLIPPAGE)
            no_maliyeti = float(prices[1]) * (1 + FEE_RATE + SLIPPAGE)
            toplam_maliyet = yes_maliyeti + no_maliyeti
            
            # Net Kâr Kontrolü
            if toplam_maliyet < 1.0:
                net_kar = 1.0 - toplam_maliyet
                yatirim_orani = kelly_kriteri(net_kar, toplam_maliyet)
                print(f"\n[FIRSAT TESPİT EDİLDİ]: {baslik}")
                print(f"Toplam Maliyet: {toplam_maliyet:.4f}$ | Net Kâr: {net_kar:.4f}$")
                print(f"Kelly Kriteri Tavsiyesi: Toplam portföyün %{yatirim_orani * 100:.2f}'sini bu emre ayırın.")

print("\nTarama tamamlandı. (Fırsat bulunamadıysa piyasa verimli çalışıyor demektir ve çıktı boş döner)")

## Simülasyon ve Stres Testi Mantığı (Backtest)

**Sonuçların Yorumlanması ve Neden Simülasyona İhtiyaç Var?**
Üstteki canlı API taramasının boş dönmesi (hiçbir fırsat bulunamaması) beklenen bir durumdur. Finansal piyasalar doğası gereği çoğunlukla etkindir (efficient) ve arbitraj fırsatları yüksek frekanslı (HFT) algoritmalar tarafından milisaniyeler içinde kapanır. Ayrıca, %0 yatırımla sonuçlanan (sıfır dönen) durumlar, algoritmanın bozuk olduğu anlamına değil; Kelly Kriterinin 0.95$ riske atıp sadece 0.05$ kazanmayı istatistiksel olarak sürdürülemez bulduğu (negatif beklenen getiri / EV) ve sermayeyi iflastan koruduğu anlamına gelir. 

Geliştirdiğimiz algoritmanın kârlı, riskli veya zarar ettirme potansiyeli olan durumlarda kasayı nasıl dinamik olarak yönettiğini kanıtlamak için aşağıda sanal piyasa koşulları (senaryolar) oluşturuyoruz.

In [ ]:
# Farklı piyasa koşullarını (stres testlerini) simüle eden veri seti
senaryolar = [
    {"isim": "Kusursuz Fırsat (Büyük Haber Akışı, Yüksek Likidite)", "maliyet": 0.80, "olasilik": 0.99},
    {"isim": "Ortalama Fırsat (Anlık Likidite Boşluğu)", "maliyet": 0.92, "olasilik": 0.95},
    {"isim": "Yüksek Riskli Fırsat (Platformun Emri Reddetme İhtimali Yüksek)", "maliyet": 0.96, "olasilik": 0.90},
    {"isim": "Negatif Beklenti (İşlem Maliyetlerinin Kârı Erittiği Durum)", "maliyet": 0.95, "olasilik": 0.95}
]

print("--- Sanal Arbitraj Senaryoları ve Portföy Yönetimi ---\n")

for s in senaryolar:
    kar = 1.0 - s["maliyet"] # 1 Dolar garanti ödemeden maliyet çıkarılıyor
    
    # Kelly Kriteri fonksiyonunu üst hücrede tanımlamıştık, burada sanal senaryolarla çağırıyoruz
    yatirim_orani = kelly_kriteri(kar, s["maliyet"], s["olasilik"])
    
    print(f"Senaryo: {s['isim']}")
    print(f"Brüt Maliyet: {s['maliyet']:.2f}$ | Net Beklenen Kâr: {kar:.2f}$ | Sistem Başarı İhtimali: %{s['olasilik']*100}")
    print(f"ALGORİTMA KARARI: Kasanın %{yatirim_orani * 100:.1f}'ini işleme dahil et.\n" + "-"*60)


## Proje Özeti ve Finansal Yorumlama

Bu proje, kantitatif finansta (quant finance) kritik olan veri mühendisliği ve risk yönetimi adımlarını tek bir akışta birleştirmektedir. Bankacılık ve finans alanında teknik bir altyapıya sahip olmak, yalnızca piyasadan veri çekmekle değil; bu veriyi maliyet odaklı filtrelemekle ve matematiksel olarak konumlandırmakla mümkündür.

**Simülasyon Çıktılarının Analizi:**
* Sistem, maliyetin 0.80$ olduğu **"Kusursuz Fırsat"** senaryosunda çok yüksek bir kâr/zarar oranı saptamış ve portföyün **%95'ini** işleme tahsis ederek getiriyi maksimize etmiştir.
* Kâr marjının daraldığı ve işlem riskinin arttığı senaryolarda sermaye tahsisi sert bir şekilde **%30'lara** düşürülmüştür.
* En önemlisi, **"Negatif Beklenti"** senaryosunda (0.95$ maliyet, %95 başarı ihtimali) algoritma yatırım oranını **%0.0** olarak belirlemiştir. Bu sıfır sonucu, sistemin beklenen getirisi negatif olan (Expected Value < 0) bir riski matematiksel olarak reddettiğinin ve iflas riskine (Risk of Ruin) karşı koruma sağladığının en net kanıtıdır.

*(Bu notebook, istatistik altyapısıyla kantitatif finans alanındaki becerileri sergilemek üzere tasarlanmıştır.)*